# BBC News Classification Project
- Author: Alex Waddel
- Date: 11/30/2025
## Overview
*Write summary here.*

In [82]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")


## Load Data

In [ ]:
DATA_DIR = Path.cwd() / "data"
train_df = pd.read_csv(DATA_DIR/'BBC News Train.csv')
test_df = pd.read_csv(DATA_DIR/'BBC News Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'BBC News Sample Solution.csv')
train_df.head()

,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business


## EDA

### Plot histogram of word count per article

In [ ]:
# Plot histogram of word count per article
train_df['Word_Count']=train_df['Text'].str.split().str.len()
sns.histplot(train_df['Word_Count']); plt.show()

The distribution of word count across the articles is roughly contained between 0-1000 words. The distribution is somewhat gaussian, but with a significant right skew.

### Plot number of articles per category in training set

In [ ]:
# Plot number of articles per category in training set
train_df['Category'].value_counts().plot(kind='bar', figsize=(8,4))
plt.title("Number of Articles per Category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.show()

This training data set is pretty well balanced between categories.

## Feature Extraction of Raw Texts

Use TF-IDF to generate features
Need blurb on NLP/TF-IDF/etc. here. include references
Used TF-IDF since it doesn't generate negative values and I wanted to use NMF

In [75]:
# Use TF-IDF to process raw texts into feature vectors
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000, # [None, 1000, 3000, 5000]
    norm='l2',
    max_df = 1.0, # [0.4, 0.6, 0.8, None]
    min_df = 1, # [1, 2, 4]
    )
X_train = tfidf.fit_transform(train_df['Text'])
X_test = tfidf.transform(test_df['Text'])
y_train = train_df['Category']

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
sorted_idx = np.argsort(X_train.mean(axis=0).A1)[::-1]
print(f"Feature Names: {feature_names[sorted_idx[:20]]}")

In [ ]:
def top_tfidf(category, n=10):
    idx = (train_df['Category'] == category).values
    vec = X_train[idx].mean(axis=0).A1
    top_idx = np.argsort(vec)[::-1][:n]
    return list(zip(feature_names[top_idx], vec[top_idx]))

# @TODO: Maybe turn these into plots for each category showing word score

top_tfidf("business")

## Matrix Factorization Unsupervised Classification Model

Use NMF
This is why

In [ ]:
# Need to build custom NMF class so I can use sklearn gridsearch later
class CustomNMFClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=5):
        self.n_components = n_components

    def fit(self, X, y):
        self.nmf = NMF(n_components=self.n_components, 
                  random_state=22)
        self.nmf.fit(X)
        W = self.nmf.transform(X_train)
        y_pred = np.argmax(W, axis=1)
        self.nmf_category_labels = {}
        categories = np.unique(y.values)
        
        # Loop through true categories and associate cluster with each
        best_perm = None
        score = 0
        for perm in itertools.permutations(range(len(categories))):
            categories_temp = np.array([categories[perm.index(ind)] for ind in y_pred])
            score_temp = np.mean(categories_temp == y.values)
            if score_temp > score:
                score = score_temp
                best_perm = perm

        # Assign mapping to dictionary
        for i in range(len(categories)):
            self.nmf_category_labels[i] = categories[best_perm.index(i)]

    def predict(self, X):
        W = self.nmf.transform(X)
        y_pred = np.argmax(W, axis=1)
        y_pred_categories = [self.nmf_category_labels[c] for c in y_pred]

        return y_pred_categories


In [ ]:
nmf = CustomNMFClassifier(n_components=5)
nmf.fit(X_train, y_train)
y_pred = nmf.predict(X_train)
acc = accuracy_score(y_train, y_pred)
print(f"Training Classification Accuracy: {acc}")


Training Classification Accuracy: 0.9187919463087248


In [ ]:
# @TODO: some visuals of outputs?

## Tune Hyperparameters of TFIDF-NMF

In [85]:
# Use grid search cross validation to fine optimal hyperparameters
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(norm="l2")),
    ("nmf", CustomNMFClassifier(n_components=5)),
])
param_grid = {
    "tfidf__max_features": [None, 1000, 3000, 5000],
    "tfidf__max_df": [0.4, 0.6, 0.8, 1.0],
    "tfidf__min_df": [1, 2, 4],
}
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
)

grid.fit(X_train, y_train)

print("Best score:", grid.best_score_)
print("Best params:", grid.best_params_)

# -------------------------------
# Evaluate on test set
# -------------------------------
y_pred = grid.best_estimator_.predict(X_train)
print(classification_report(y_train, y_pred))


ValueError: 
All the 144 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
144 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Code\.venv\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Code\.venv\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Code\.venv\lib\site-packages\sklearn\pipeline.py", line 655, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "c:\Code\.venv\lib\site-packages\sklearn\pipeline.py", line 589, in _fit
    X, fitted_transformer = fit_transform_one_cached(
  File "c:\Code\.venv\lib\site-packages\joblib\memory.py", line 326, in __call__
   

## Supervised Logistic Regression

In [ ]:
clf_sup = LogisticRegression(max_iter=500)
clf_sup.fit(X_train, y_train)
print(clf_sup.score(X_train, y_train))

## Submission

In [ ]:
submission = sample_sub.copy()
submission['category'] = clf_unsup.predict(W_test)
submission.to_csv('submission.csv', index=False)
submission.head()